In [ ]:
!python3 -m pip install -U python-woc
# please clear the output of this cell in your notebook before checking it in

In [22]:
from woc.remote import WocMapsRemote
# this creates access object
woc = WocMapsRemote( base_url="https://worldofcode.org/api/") #, api_key="woc-XXXXXXXXXXXXXX-YYYYYYYYYYYYYY" )

# Now for each of the ten projects you were assigned get commits
# (note that woc ids have to be lowercase)
prj = 'user2589_MiniCMS'.lower()
commits = woc.get_values('p2c', prj.lower())
#check for errors
#perhaps save the list (if no errors) so you do not need to retrieve them again
import pandas as pd
df = pd.DataFrame(commits, columns=["sha1"])
df.to_csv(prj+'_commits.csv')
print(df.shape)
df.head(1)

(69, 1)


,sha1
0,05cf84081b63cda822ee407e688269b494a642de


In [20]:
#If the number of commits its not very large, you can try to get them all at the same time,
# otherwise batch, in 300 commits or so
info = woc.show_content_many('commit',df['sha1'].to_list())
# check for errors (second element, returned value in the first)
print(info[1])
#empty "{}" means no error
# does it have the same number of commits?
print(len(info[0]))

{}
69


In [27]:
# what does the first record looks like?
# commit  parents
v = info[0]['05cf84081b63cda822ee407e688269b494a642de']
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

['85575a7546aef8dfbee6fff30d83c24d24a5454a', ['4dffda766eba4f4edc31eb0b7691cc75d7775de0'], ['Pavel Puchkin <neoascetic@gmail.com>', '1337338456', '+1100'], ['Pavel Puchkin <neoascetic@gmail.com>', '1337338456', '+1100'], 'Markdown editor integrated\n']


In [29]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns 'commit, author, time, message'
dfinf = pd.DataFrame(columns=['commit', 'author', 'time', 'message'])
for k in info[0].keys():
  row = pd.Series({'commit': k, 'author': info[0][k][2][0], 'time': int(info[0][k][2][1]), 'message':info[0][k][3]})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [31]:
# check if it has the right content
dfinf.head(1)

,commit,author,time,message
0,05cf84081b63cda822ee407e688269b494a642de,Pavel Puchkin <neoascetic@gmail.com>,1337338456,"[Pavel Puchkin <neoascetic@gmail.com>, 1337338..."


In [32]:
dfinf.to_csv(prj+'_info.csv', index=False)